In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer 

In [ ]:
# Install dependencies as needed:
%pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "fake_job_postings.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "shivamb/real-or-fake-fake-jobposting-prediction",
  file_path,
)

## Load data
### 2. Combine the text columns into one column called "text"

In [ ]:
df["text"] = (
    df["title"].fillna("") + " "
    + df["company_profile"].fillna("") + " "
    + df["description"].fillna("") + " "
    + df["requirements"].fillna("")
)

df.head()

df.drop(columns = ['title', 'company_profile', 'description', 'requirements', 'job_id', 'location','department'
                   , 'benefits', 'salary_range', 'function', 'industry'], inplace = True)


In [ ]:
df.head(5)
df.isnull().sum()

### 3. Fill missing values in categorical columns

In [ ]:
cat_col = ['employment_type', 'required_experience', 'required_education']
for col in cat_col:
    df[col] = df[col].fillna('Unknown')

num_col = ['telecommuting', 'has_company_logo', 'has_questions']

### 4. Train / test split (stratify because fraud is rare)

In [ ]:
X = df[['text'] + cat_col + num_col]
y = df['fraudulent']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 5. ColumnTransformer: different processing for different columns

In [ ]:
preprocessor = ColumnTransformer(transformers = [
    ('text', CountVectorizer(), 'text'),
    ('cat', OneHotEncoder(handle_unknown ='ignore'), cat_col),
    ('num', 'passthrough', num_col)
])



### 6. Pipeline: preprocessing + model

In [ ]:
model = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter = 1000, class_weight = 'balanced'))
])
model.fit(x_train, y_train)


model_2 = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter = 1000, class_weight = 'balanced'))
])
model.fit(x_train, y_train)

### 7. Evaluate

In [ ]:
y_pred = model.predict(x_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

### 8. Save

In [ ]:
joblib.dump(model, 'fake_job_post_detector_model.pkl')
print("Model saved as 'fake_job_post_detector_model.pkl'")